In [ ]:
import os
import re
import gc
import copy
import time
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from transformers import BertModel, AutoModelForCausalLM, AutoTokenizer
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/CMU-MOSI/Processed/Processed/unaligned_50.pkl"
SAVE_DIR = "/content/predictions"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(DATA_PATH, "rb") as f:
    data = pickle.load(f, encoding="latin1")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch: {torch.__version__}")
print(f"Splits: Train={len(data['train']['regression_labels'])}, Valid={len(data['valid']['regression_labels'])}, Test={len(data['test']['regression_labels'])}")

In [ ]:
def set_seed(seed=42, deterministic=False):
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic
    torch.use_deterministic_algorithms(deterministic, warn_only=True)

def get_feature_loaders(data_dict, batch_size=128, seed=1115, use_generator=True):
    def _make_dataset(split):
        t = torch.tensor(np.nan_to_num(np.asarray(data_dict[split]["text"], dtype=np.float32)))
        a = torch.tensor(np.nan_to_num(np.asarray(data_dict[split]["audio"], dtype=np.float32)))
        v = torch.tensor(np.nan_to_num(np.asarray(data_dict[split]["vision"], dtype=np.float32)))
        y = torch.tensor(np.nan_to_num(np.asarray(data_dict[split]["regression_labels"], dtype=np.float32))).reshape(-1, 1)
        return TensorDataset(t, a, v, y)

    gen = torch.Generator().manual_seed(seed) if use_generator else None
    train_loader = DataLoader(_make_dataset("train"), batch_size=batch_size, shuffle=True, generator=gen)
    valid_loader = DataLoader(_make_dataset("valid"), batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(_make_dataset("test"),  batch_size=batch_size, shuffle=False)
    return train_loader, valid_loader, test_loader

class BertMOSIDataset(Dataset):
    def __init__(self, split_data):
        self.text_bert = torch.as_tensor(np.asarray(split_data["text_bert"]), dtype=torch.long)
        self.audio = torch.as_tensor(np.asarray(split_data["audio"]), dtype=torch.float32)
        self.vision = torch.as_tensor(np.asarray(split_data["vision"]), dtype=torch.float32)
        self.labels = torch.as_tensor(np.asarray(split_data["regression_labels"]), dtype=torch.float32).reshape(-1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.text_bert[idx], self.audio[idx], self.vision[idx], self.labels[idx]

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)

    true_has0 = (y_true >= 0).astype(int)
    pred_has0 = (y_pred >= 0).astype(int)
    has0_acc = accuracy_score(true_has0, pred_has0) * 100
    has0_f1  = f1_score(true_has0, pred_has0, average="binary", zero_division=0) * 100

    mask = y_true != 0
    true_non0 = (y_true[mask] > 0).astype(int)
    pred_non0 = (y_pred[mask] > 0).astype(int)
    non0_acc = accuracy_score(true_non0, pred_non0) * 100
    non0_f1  = f1_score(true_non0, pred_non0, average="binary", zero_division=0) * 100

    mae  = mean_absolute_error(y_true, y_pred)
    corr = pearsonr(y_true, y_pred)[0] if (np.std(y_true) > 0 and np.std(y_pred) > 0) else np.nan

    return {
        "Has0_Acc2": has0_acc,
        "Has0_F1": has0_f1,
        "Non0_Acc2": non0_acc,
        "Non0_F1": non0_f1,
        "MAE": mae,
        "Corr": corr
    }

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        if x.dim() == 2:
            return x.float()
        weights = torch.softmax(self.attention(x.float()).squeeze(-1), dim=1)
        return torch.sum(x.float() * weights.unsqueeze(-1), dim=1)

# 1. Early Fusion (LFDNN)
class LFDNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_net = nn.Sequential(nn.Linear(768, 128), nn.ReLU(), nn.Dropout(0.2))
        self.audio_net = nn.Sequential(nn.Linear(5, 16), nn.ReLU(), nn.Dropout(0.2))
        self.vision_net = nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Dropout(0.2))
        self.fusion = nn.Sequential(nn.Linear(272, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))

    def forward(self, text, audio, vision):
        combined = torch.cat([
            self.text_net(text.mean(dim=1)),
            self.audio_net(audio.mean(dim=1)),
            self.vision_net(vision.mean(dim=1))
        ], dim=1)
        return self.fusion(combined)

# 2. Late Fusion (LF-DNN)
class SubNet(nn.Module):
    def __init__(self, in_size, hidden_size, dropout):
        super().__init__()
        self.norm = nn.BatchNorm1d(in_size)
        self.drop = nn.Dropout(dropout)
        self.linear_1 = nn.Linear(in_size, hidden_size)
        self.linear_2 = nn.Linear(hidden_size, hidden_size)

    def forward(self, x):
        return self.linear_2(F.relu(self.linear_1(self.drop(self.norm(x)))))

class TextSubNet(nn.Module):
    def __init__(self, in_size, hidden_size, output_size, dropout):
        super().__init__()
        self.rnn = nn.LSTM(in_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.linear_1 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        _, (hidden, _) = self.rnn(x)
        return self.linear_1(self.dropout(hidden[-1]))

class LateFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.audio_subnet = SubNet(5, 16, 0.1)
        self.video_subnet = SubNet(20, 16, 0.1)
        self.text_subnet = TextSubNet(768, 128, 64, 0.1)
        self.post_fusion = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(96, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, text, audio, vision):
        audio_h = self.audio_subnet(audio.mean(dim=1))
        vision_h = self.video_subnet(vision.mean(dim=1))
        text_h = self.text_subnet(text)
        return self.post_fusion(torch.cat([audio_h, vision_h, text_h], dim=-1))

# 3. Hybrid Fusion (CENET)
class CrossModalEnhancement(nn.Module):
    def __init__(self, hidden_size=768, audio_dim=5, vision_dim=20, num_heads=8, dropout=0.1):
        super().__init__()
        self.audio_proj = nn.Linear(audio_dim, hidden_size)
        self.vision_proj = nn.Linear(vision_dim, hidden_size)
        self.audio_att = nn.MultiheadAttention(hidden_size, num_heads, dropout=dropout, batch_first=True)
        self.vision_att = nn.MultiheadAttention(hidden_size, num_heads, dropout=dropout, batch_first=True)
        self.fusion = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size)
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text_features, audio, vision):
        a_ctx, _ = self.audio_att(text_features, self.audio_proj(audio), self.audio_proj(audio), need_weights=False)
        v_ctx, _ = self.vision_att(text_features, self.vision_proj(vision), self.vision_proj(vision), need_weights=False)
        enhanced = self.fusion(torch.cat([a_ctx, v_ctx], dim=-1))
        return self.norm(text_features + self.dropout(enhanced))

class CENET(nn.Module):
    def __init__(self, model_name="bert-base-uncased", injection_index=1, dropout=0.2):
        super().__init__()
        self.injection_index = injection_index
        self.bert = BertModel.from_pretrained(model_name, add_pooling_layer=False)
        hidden_size = self.bert.config.hidden_size
        self.ce = CrossModalEnhancement(hidden_size=hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, text_bert, audio, vision):
        target_layer = self.bert.encoder.layer[self.injection_index - 1]
        def inject_ce(module, inputs, output):
            hidden = output if isinstance(output, tuple) else output
            enhanced = self.ce(hidden, audio, vision)
            return (enhanced,) + output[1:] if isinstance(output, tuple) else enhanced

        handle = target_layer.register_forward_hook(inject_ce)
        try:
            outputs = self.bert(
                input_ids=text_bert[:, 0, :],
                attention_mask=text_bert[:, 1, :],
                token_type_ids=text_bert[:, 2, :],
                return_dict=True
            )
        finally:
            handle.remove()

        return self.regressor(self.dropout(outputs.last_hidden_state[:, 0, :])).squeeze(-1)

print("Baseline models defined.")
# 4. Early Fusion + Multi-Step CoT
class EarlyFusionCoT(nn.Module):
    def __init__(self, hidden_dim=128, dropout=0.25):
        super().__init__()
        self.text_pool = TemporalAttention(768, 64)
        self.audio_pool = TemporalAttention(5, 64)
        self.vision_pool = TemporalAttention(20, 64)
        self.text_enc = nn.Sequential(nn.Linear(768, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.audio_enc = nn.Sequential(nn.Linear(5, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.vision_enc = nn.Sequential(nn.Linear(20, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.mod_att = nn.Sequential(nn.Linear(hidden_dim, 64), nn.Tanh(), nn.Linear(64, 1))
        self.fusion = nn.Sequential(nn.Linear(hidden_dim * 3, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.reasoning_cell = nn.GRUCell(hidden_dim, hidden_dim)
        self.step_norm = nn.LayerNorm(hidden_dim)
        self.step_heads = nn.ModuleList([nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(), nn.Linear(64, 1)) for _ in range(3)])
        self.final_head = nn.Sequential(
            nn.Linear(hidden_dim * 3, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.15), nn.Linear(64, 1)
        )

    def forward(self, text, audio, vision):
        tf = self.text_enc(self.text_pool(text))
        af = self.audio_enc(self.audio_pool(audio))
        vf = self.vision_enc(self.vision_pool(vision))
        mod_feats = torch.stack([tf, af, vf], dim=1)
        weights = torch.softmax(self.mod_att(mod_feats).squeeze(-1), dim=1)
        fused = self.fusion((mod_feats * weights.unsqueeze(-1)).reshape(tf.size(0), -1))

        hidden = fused
        step_preds = []
        for idx, feat in enumerate([tf, af, vf]):
            hidden = self.step_norm(hidden + self.reasoning_cell(feat, hidden))
            step_preds.append(self.step_heads[idx](hidden).squeeze(-1))

        final_pred = self.final_head(torch.cat([hidden, fused, tf], dim=-1)).squeeze(-1)
        return final_pred, step_preds, weights

# 5. Late Fusion + PEAR-CoT + CMFF
class PEARCoT_CMFF(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.text_enc = nn.Sequential(nn.Linear(768, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(0.20))
        self.audio_enc = nn.Sequential(nn.Linear(5, 64), nn.LayerNorm(64), nn.GELU(), nn.Linear(64, d), nn.LayerNorm(d), nn.GELU())
        self.vision_enc = nn.Sequential(nn.Linear(20, 64), nn.LayerNorm(64), nn.GELU(), nn.Linear(64, d), nn.LayerNorm(d), nn.GELU())
        self.text_att = TemporalAttention(d, d // 2)
        self.audio_att = TemporalAttention(d, d // 2)
        self.vision_att = TemporalAttention(d, d // 2)
        self.ta_att = nn.MultiheadAttention(d, 4, dropout=0.15, batch_first=True)
        self.tv_att = nn.MultiheadAttention(d, 4, dropout=0.15, batch_first=True)
        self.audio_gate = nn.Sequential(nn.Linear(d * 2, d), nn.Sigmoid())
        self.vision_gate = nn.Sequential(nn.Linear(d * 2, d), nn.Sigmoid())
        self.cmff = nn.Sequential(nn.Linear(d * 5, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.25), nn.Linear(256, d), nn.LayerNorm(d), nn.GELU())
        self.reason_query = nn.Sequential(nn.Linear(d, d), nn.Tanh())
        self.reason_cell = nn.GRUCell(d, d)
        self.reason_norm = nn.LayerNorm(d)
        self.reason_gate = nn.Sequential(nn.Linear(d * 2, d), nn.Sigmoid())
        self.regression = nn.Sequential(nn.Linear(d * 2, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.25), nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.15), nn.Linear(64, 1))

    def forward(self, text, audio, vision):
        t, a, v = self.text_enc(text), self.audio_enc(audio), self.vision_enc(vision)
        tp, ap, vp = self.text_att(t), self.audio_att(a), self.vision_att(v)
        ta = self.text_att(self.ta_att(t, a, a, need_weights=False)[0])
        tv = self.text_att(self.tv_att(t, v, v, need_weights=False)[0])
        ga = self.audio_gate(torch.cat([tp, ta], dim=1))
        gv = self.vision_gate(torch.cat([tp, tv], dim=1))
        fused = self.cmff(torch.cat([tp, ga * ta + (1 - ga) * ap, gv * tv + (1 - gv) * vp, ta, tv], dim=1))

        state = fused
        for _ in range(3):
            state = self.reason_norm(self.reason_cell(self.reason_query(state), fused))
        gate = self.reason_gate(torch.cat([fused, state], dim=1))
        final_feat = gate * state + (1 - gate) * fused
        return self.regression(torch.cat([fused, final_feat], dim=1))

print("CoT neural models defined.")

In [ ]:
def train_simple_baseline(model, name, train_loader, valid_loader, test_loader, save_path, patience=10):
    print(f"--- Training {name} ---")
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)
    best_loss, best_state, wait = float("inf"), None, 0

    for epoch in range(1, 101):
        model.train()
        train_loss, n_train = 0.0, 0
        for t, a, v, y in train_loader:
            t, a, v, y = t.to(device), a.to(device), v.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(t, a, v), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * y.size(0)
            n_train += y.size(0)

        model.eval()
        val_loss, n_val = 0.0, 0
        with torch.no_grad():
            for t, a, v, y in valid_loader:
                t, a, v, y = t.to(device), a.to(device), v.to(device), y.to(device)
                val_loss += criterion(model(t, a, v), y).item() * y.size(0)
                n_val += y.size(0)
        val_loss /= n_val

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d}/100 | Train MAE: {train_loss/n_train:.4f} | Val MAE: {val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for t, a, v, y in test_loader:
            preds.extend(model(t.to(device), a.to(device), v.to(device)).cpu().numpy().reshape(-1))
            labels.extend(y.numpy().reshape(-1))
    np.savez(save_path, y_true=np.asarray(labels), y_pred=np.asarray(preds))
    print(f"=> {name} finished | Best Val MAE: {best_loss:.4f}\n")
    return best_loss

# 3.1 Train Early Fusion (LFDNN)
set_seed(1115, deterministic=False)
train_loader, valid_loader, test_loader = get_feature_loaders(data, batch_size=128, seed=1115, use_generator=True)
early_model = LFDNN().to(device)
train_simple_baseline(early_model, "Early Fusion (LFDNN)", train_loader, valid_loader, test_loader, f"{SAVE_DIR}/early_fusion_predictions.npz", patience=10)

# 3.2 Train Late Fusion (LF-DNN)
late_model = LateFusionModel().to(device)
train_simple_baseline(late_model, "Late Fusion (LF-DNN)", train_loader, valid_loader, test_loader, f"{SAVE_DIR}/late_fusion_predictions.npz", patience=8)

# 3.3 Train Hybrid Fusion (CENET)
print("--- Training Hybrid Fusion (CENET) ---")
set_seed(42, deterministic=True)
g_cenet = torch.Generator().manual_seed(42)
cenet_train_loader = DataLoader(BertMOSIDataset(data["train"]), batch_size=16, shuffle=True, generator=g_cenet)
cenet_valid_loader = DataLoader(BertMOSIDataset(data["valid"]), batch_size=16, shuffle=False)
cenet_test_loader  = DataLoader(BertMOSIDataset(data["test"]),  batch_size=16, shuffle=False)

cenet_model = CENET(injection_index=1, dropout=0.2).to(device)
criterion_mse = nn.MSELoss()
optimizer_cenet = torch.optim.AdamW(cenet_model.parameters(), lr=2e-5, weight_decay=1e-4)
scheduler_cenet = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_cenet, mode="min", factor=0.5, patience=2)

def eval_cenet(loader):
    cenet_model.eval()
    loss_sum, count, preds, labels = 0.0, 0, [], []
    with torch.no_grad():
        for tb, a, v, y in loader:
            tb, a, v, y = tb.to(device), a.to(device), v.to(device), y.to(device)
            out = cenet_model(tb, a, v)
            loss_sum += criterion_mse(out, y).item() * y.size(0)
            count += y.size(0)
            preds.extend(out.cpu().numpy())
            labels.extend(y.cpu().numpy())
    return loss_sum / count, np.asarray(preds), np.asarray(labels)

best_cenet_mse, best_cenet_state = float("inf"), None
for epoch in range(1, 21):
    cenet_model.train()
    train_sum, train_cnt = 0.0, 0
    for tb, a, v, y in cenet_train_loader:
        tb, a, v, y = tb.to(device), a.to(device), v.to(device), y.to(device)
        optimizer_cenet.zero_grad(set_to_none=True)
        loss = criterion_mse(cenet_model(tb, a, v), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cenet_model.parameters(), 1.0)
        optimizer_cenet.step()
        train_sum += loss.item() * y.size(0)
        train_cnt += y.size(0)

    val_mse, _, _ = eval_cenet(cenet_valid_loader)
    scheduler_cenet.step(val_mse)
    status = ""
    if val_mse < best_cenet_mse:
        best_cenet_mse = val_mse
        best_cenet_state = copy.deepcopy(cenet_model.state_dict())
        status = " [BEST]"
    print(f"Epoch {epoch:02d}/20 | Train MSE: {train_sum/train_cnt:.4f} | Val MSE: {val_mse:.4f}{status}")

cenet_model.load_state_dict(best_cenet_state)
_, valid_preds, valid_labels = eval_cenet(cenet_valid_loader)
_, cenet_predictions, cenet_labels = eval_cenet(cenet_test_loader)

np.savez(f"{SAVE_DIR}/cenet_valid_predictions.npz", y_true=valid_labels, y_pred=valid_preds)
np.savez(f"{SAVE_DIR}/cenet_predictions.npz", y_true=cenet_labels, y_pred=cenet_predictions)
print(f"=> Hybrid CENET finished | Best Val MSE: {best_cenet_mse:.4f}")

In [ ]:
# 4.1 Early Fusion + Multi-Step CoT
print("--- Training Early Fusion + CoT ---")
set_seed(1115, deterministic=False)
train_loader, valid_loader, test_loader = get_feature_loaders(data, batch_size=128, seed=1115, use_generator=False)
early_cot_model = EarlyFusionCoT(hidden_dim=128, dropout=0.25).to(device)
criterion_huber = nn.SmoothL1Loss(beta=1.0)
opt_early_cot = torch.optim.AdamW(early_cot_model.parameters(), lr=1e-4, weight_decay=1e-3)
sch_early_cot = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_early_cot, mode="min", factor=0.5, patience=3, min_lr=1e-6)

best_val_mae, best_state, wait = float("inf"), None, 0
for epoch in range(1, 101):
    early_cot_model.train()
    train_loss, n_train = 0.0, 0
    for t, a, v, y in train_loader:
        t, a, v, target = t.to(device), a.to(device), v.to(device), y.to(device).reshape(-1)
        opt_early_cot.zero_grad(set_to_none=True)
        final_pred, step_preds, _ = early_cot_model(t, a, v)
        loss = criterion_huber(final_pred, target) + 0.10 * (sum(criterion_huber(p, target) for p in step_preds) / 3.0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(early_cot_model.parameters(), 1.0)
        opt_early_cot.step()
        train_loss += loss.item() * target.size(0)
        n_train += target.size(0)

    early_cot_model.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for t, a, v, y in valid_loader:
            t, a, v, target = t.to(device), a.to(device), v.to(device), y.to(device).reshape(-1)
            pred, _, _ = early_cot_model(t, a, v)
            val_loss += criterion_huber(pred, target).item() * target.size(0)
            n_val += target.size(0)
    val_mae = val_loss / n_val
    sch_early_cot.step(val_mae)

    status = ""
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_state = copy.deepcopy(early_cot_model.state_dict())
        wait = 0
        status = " [BEST]"
    else:
        wait += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}/100 | Train Loss: {train_loss/n_train:.4f} | Val MAE: {val_mae:.4f}{status}")

    if wait >= 12:
        print(f"Early stopping at epoch {epoch}")
        break

early_cot_model.load_state_dict(best_state)
early_cot_model.eval()
preds, labels = [], []
with torch.no_grad():
    for t, a, v, y in test_loader:
        p, _, _ = early_cot_model(t.to(device), a.to(device), v.to(device))
        preds.extend(p.cpu().numpy().reshape(-1))
        labels.extend(y.numpy().reshape(-1))
np.savez(f"{SAVE_DIR}/early_fusion_cot_predictions.npz", y_true=np.asarray(labels), y_pred=np.asarray(preds))
print(f"=> Early Fusion + CoT finished | Best Val MAE: {best_val_mae:.4f}\n")

# 4.2 Late Fusion + PEAR-CoT + CMFF
print("--- Training Late Fusion + PEAR-CoT + CMFF ---")
set_seed(42, deterministic=True)
train_loader_32, valid_loader_32, test_loader_32 = get_feature_loaders(data, batch_size=32, seed=42, use_generator=False)
pear_model = PEARCoT_CMFF().to(device)
criterion_pear = nn.SmoothL1Loss(beta=0.8)
opt_pear = torch.optim.AdamW(pear_model.parameters(), lr=3e-4, weight_decay=5e-3)
sch_pear = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_pear, mode="max", factor=0.5, patience=3, min_lr=1e-6)

def predict_pear(loader):
    pear_model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for t, a, v, y in loader:
            out = pear_model(t.to(device), a.to(device), v.to(device))
            preds.extend(out.cpu().numpy().reshape(-1))
            labels.extend(y.numpy().reshape(-1))
    return np.asarray(preds), np.asarray(labels)

def find_best_threshold(y_true, y_pred, nonzero=False):
    mask = (y_true != 0) if nonzero else np.ones(len(y_true), dtype=bool)
    true = (y_true[mask] >= 0).astype(int)
    pred = y_pred[mask]
    best_th, best_sc = 0.0, -1.0
    for th in np.arange(-0.40, 0.401, 0.005):
        predicted = (pred >= th).astype(int)
        sc = 0.75 * accuracy_score(true, predicted) + 0.25 * f1_score(true, predicted, zero_division=0)
        if sc > best_sc:
            best_sc, best_th = sc, float(th)
    return best_th

best_score, best_pear_state, wait = -1.0, None, 0
for epoch in range(1, 81):
    pear_model.train()
    train_loss, n_train = 0.0, 0
    for t, a, v, y in train_loader_32:
        t, a, v, y = t.to(device), a.to(device), v.to(device), y.to(device)
        opt_pear.zero_grad(set_to_none=True)
        loss = criterion_pear(pear_model(t, a, v), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pear_model.parameters(), 1.0)
        opt_pear.step()
        train_loss += loss.item() * y.size(0)
        n_train += y.size(0)

    vp, vt = predict_pear(valid_loader_32)
    th_h = find_best_threshold(vt, vp, nonzero=False)
    th_n = find_best_threshold(vt, vp, nonzero=True)
    h_acc = accuracy_score((vt >= 0).astype(int), (vp >= th_h).astype(int))
    h_f1  = f1_score((vt >= 0).astype(int), (vp >= th_h).astype(int), zero_division=0)
    mask = vt != 0
    n_acc = accuracy_score((vt[mask] >= 0).astype(int), (vp[mask] >= th_n).astype(int))
    n_f1  = f1_score((vt[mask] >= 0).astype(int), (vp[mask] >= th_n).astype(int), zero_division=0)
    score = 0.35 * h_acc + 0.35 * n_acc + 0.15 * h_f1 + 0.15 * n_f1
    sch_pear.step(score)

    status = ""
    if score > best_score:
        best_score = score
        best_pear_state = copy.deepcopy(pear_model.state_dict())
        wait = 0
        status = " [BEST]"
    else:
        wait += 1

    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/80 | Loss: {train_loss/n_train:.4f} | Has0: {h_acc*100:.2f}% | Non0: {n_acc*100:.2f}% | Score: {score:.4f}{status}")

    if wait >= 14:
        print(f"Early stopping at epoch {epoch}")
        break

pear_model.load_state_dict(best_pear_state)
test_p, test_t = predict_pear(test_loader_32)
np.savez(f"{SAVE_DIR}/pear_cot_cmff_optimized_predictions.npz", y_true=test_t, y_pred=test_p)
print(f"=> Late Fusion + PEAR-CoT + CMFF finished | Best Val Score: {best_score:.4f}")

In [ ]:
# Free previous neural models from GPU memory before loading Qwen2.5-7B
for var_name in ["early_model", "late_model", "cenet_model", "early_cot_model", "pear_model"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {LLM_NAME}...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
llm_tokenizer.padding_side = "left"

llm_model = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map="auto")
llm_model.eval()
print(f"Successfully loaded {LLM_NAME}")

def make_prompt(text, audio, vision, initial_score):
    return f"""Analyze sentiment of this CMU-MOSI utterance.

Text: {str(text)[:1500]}

CENET initial sentiment score: {initial_score:.3f}
Audio intensity: {np.mean(np.abs(audio)):.3f}
Visual intensity: {np.mean(np.abs(vision)):.3f}

Sentiment scale:
-3 = very negative
0 = neutral
+3 = very positive

Consider the actual meaning, context, sarcasm and emotional tone.

Return ONLY one numerical sentiment score between -3 and +3.
Do not explain.
Do not write analysis.
Do not use Markdown.

Answer:"""

def extract_score(response):
    response = response.strip()
    for pattern in [r"SCORE\s*[:=]\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))", r"FINAL\s*[:=]\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))"]:
        match = re.search(pattern, response, re.IGNORECASE)
        if match and np.isfinite(float(match.group(1))) and -3 <= float(match.group(1)) <= 3:
            return float(match.group(1))
    match = re.fullmatch(r"\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*", response)
    if match and np.isfinite(float(match.group(1))) and -3 <= float(match.group(1)) <= 3:
        return float(match.group(1))
    return np.nan

def run_cot_inference(texts, audio, vision, predictions, split_name, batch_size=8):
    n = len(predictions)
    scores = np.full(n, np.nan)
    print(f"\n--- Running CoT Inference on {split_name} ({n} samples) ---")
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        prompts = [
            llm_tokenizer.apply_chat_template(
                [{"role": "user", "content": make_prompt(texts[i], audio[i], vision[i], predictions[i])}],
                tokenize=False, add_generation_prompt=True
            )
            for i in range(start, end)
        ]
        inputs = llm_tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=768).to(llm_model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            outputs = llm_model.generate(
                **inputs, max_new_tokens=16, do_sample=False, num_beams=1,
                pad_token_id=llm_tokenizer.pad_token_id, eos_token_id=llm_tokenizer.eos_token_id
            )
        for j in range(len(prompts)):
            resp = llm_tokenizer.decode(outputs[j, input_len:], skip_special_tokens=True)
            scores[start + j] = extract_score(resp)

        if end % 40 == 0 or end == n:
            print(f"{split_name}: {end}/{n} processed")

        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    valid_cnt = int(np.isfinite(scores).sum())
    print(f"{split_name} valid CoT scores: {valid_cnt}/{n} ({valid_cnt/n*100:.2f}%)")
    return scores

def fuse_predictions(cenet, cot, beta):
    fused = cenet.copy()
    valid = np.isfinite(cot)
    fused[valid] = (1 - beta) * cenet[valid] + beta * cot[valid]
    return fused

# Generate CoT scores on Validation and optimize Beta & Threshold
cot_val = run_cot_inference(data["valid"]["raw_text"], np.asarray(data["valid"]["audio"]), np.asarray(data["valid"]["vision"]), valid_preds, "VALIDATION")

best_val_acc = accuracy_score(valid_labels >= 0, valid_preds >= 0)
best_beta, best_threshold = 0.0, 0.0
val_records = []

for beta in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.0]:
    fused = fuse_predictions(valid_preds, cot_val, beta)
    for th in np.arange(-0.30, 0.301, 0.025):
        acc = accuracy_score(valid_labels >= 0, fused >= th)
        val_records.append({"Beta": beta, "Threshold": round(float(th), 4), "Validation_Acc2": acc})
        if acc > best_val_acc:
            best_val_acc, best_beta, best_threshold = acc, beta, float(th)

pd.DataFrame(val_records).to_csv("/content/contrastive_cot_validation_results.csv", index=False)
print(f"\nOptimal Validation Config -> Beta: {best_beta}, Threshold: {best_threshold:.3f}, Val Acc-2: {best_val_acc*100:.2f}%")

# Generate CoT scores on Test and apply fixed configuration
cot_test = run_cot_inference(data["test"]["raw_text"], np.asarray(data["test"]["audio"]), np.asarray(data["test"]["vision"]), cenet_predictions, "TEST")
hybrid_test_preds = fuse_predictions(cenet_predictions, cot_test, best_beta) - best_threshold

np.savez(
    f"{SAVE_DIR}/cenet_cot_predictions.npz",
    y_true=cenet_labels, y_pred=hybrid_test_preds,
    cenet_preds=cenet_predictions, cot_scores=cot_test,
    best_beta=best_beta, best_threshold=best_threshold
)
print("=> Hybrid CENET + Contrastive CoT completed!")

In [ ]:
methods_map = [
    ("Early Fusion (LFDNN)",      f"{SAVE_DIR}/early_fusion_predictions.npz"),
    ("Early Fusion + CoT",        f"{SAVE_DIR}/early_fusion_cot_predictions.npz"),
    ("Late Fusion (LF-DNN)",      f"{SAVE_DIR}/late_fusion_predictions.npz"),
    ("Late Fusion + CoT",         f"{SAVE_DIR}/pear_cot_cmff_optimized_predictions.npz"),
    ("Hybrid CENET",              f"{SAVE_DIR}/cenet_predictions.npz"),
    ("Hybrid CENET + CoT (Ours)", f"{SAVE_DIR}/cenet_cot_predictions.npz")
]

summary_rows = []
for name, path in methods_map:
    npz = np.load(path)
    metrics = compute_metrics(npz["y_true"], npz["y_pred"])
    summary_rows.append({"Model": name, **metrics})

df_results = pd.DataFrame(summary_rows)[["Model", "Has0_Acc2", "Has0_F1", "Non0_Acc2", "Non0_F1", "Corr"]]

display(
    df_results.style
    .format({"Has0_Acc2": "{:.2f}%", "Has0_F1": "{:.2f}%", "Non0_Acc2": "{:.2f}%", "Non0_F1": "{:.2f}%", "Corr": "{:.4f}"})
    .hide(axis="index")
    .set_properties(**{"text-align": "center", "padding": "10px", "font-size": "13px", "border": "1px solid #D3D3D3"})
    .set_properties(subset=["Model"], **{"text-align": "left", "font-weight": "bold"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#1F4E78"), ("color", "white"), ("text-align", "center"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "tr:last-child", "props": [("background-color", "#E8F5E9"), ("font-weight", "bold")]}
    ])
)

df_results.to_csv("/content/paper_5_metrics_results.csv", index=False, encoding="utf-8-sig")
df_results.to_excel("/content/paper_5_metrics_results.xlsx", index=False)
print("Saved results table to '/content/paper_5_metrics_results.csv' and '.xlsx'\n")

# Reusable plotting function for Acc-2 and F1-Score
def plot_metric_comparison(base_scores, cot_scores, ylabel, y_lim, out_prefix):
    categories = [
        "Early Fusion\n(Has0)", "Early Fusion\n(Non0)",
        "Late Fusion\n(Has0)",  "Late Fusion\n(Non0)",
        "Hybrid CENET\n(Has0)", "Hybrid CENET\n(Non0)"
    ]
    plt.rcParams["font.sans-serif"] = "DejaVu Sans"
    plt.rcParams["axes.edgecolor"] = "#333333"
    plt.rcParams["axes.linewidth"] = 0.8

    fig, ax = plt.subplots(figsize=(11, 5.0), dpi=300)
    x = np.arange(len(categories))
    width = 0.36

    r1 = ax.bar(x - width/2, base_scores, width, label="Baseline", color="#6BAED6", edgecolor="#2B5C8F", linewidth=0.9, zorder=3)
    r2 = ax.bar(x + width/2, cot_scores,  width, label="+ CoT (Ours)", color="#66C2A5", edgecolor="#1B6B50", linewidth=0.9, zorder=3)

    ax.set_ylabel(ylabel, fontsize=11.5, fontweight="bold", color="#111111")
    ax.set_ylim(*y_lim)
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=10.5, fontweight="bold", color="#222222")
    ax.tick_params(axis="y", labelsize=10, colors="#333333")
    ax.yaxis.grid(True, linestyle="--", alpha=0.5, color="#CCCCCC", zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for rects, color in [(r1, "#2B5C8F"), (r2, "#145A32")]:
        for rect in rects:
            h = rect.get_height()
            ax.annotate(f"{h:.2f}%", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 4),
                        textcoords="offset points", ha="center", va="bottom", fontsize=9.5, fontweight="bold", color=color)

    ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#DDDDDD", fontsize=10.5, framealpha=0.95)
    plt.tight_layout()
    plt.savefig(f"/content/{out_prefix}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"/content/{out_prefix}.pdf", bbox_inches="tight")
    plt.show()

acc_base = [df_results.loc[0, "Has0_Acc2"], df_results.loc[0, "Non0_Acc2"], df_results.loc[2, "Has0_Acc2"], df_results.loc[2, "Non0_Acc2"], df_results.loc[4, "Has0_Acc2"], df_results.loc[4, "Non0_Acc2"]]
acc_cot  = [df_results.loc[1, "Has0_Acc2"], df_results.loc[1, "Non0_Acc2"], df_results.loc[3, "Has0_Acc2"], df_results.loc[3, "Non0_Acc2"], df_results.loc[5, "Has0_Acc2"], df_results.loc[5, "Non0_Acc2"]]

f1_base  = [df_results.loc[0, "Has0_F1"],   df_results.loc[0, "Non0_F1"],   df_results.loc[2, "Has0_F1"],   df_results.loc[2, "Non0_F1"],   df_results.loc[4, "Has0_F1"],   df_results.loc[4, "Non0_F1"]]
f1_cot   = [df_results.loc[1, "Has0_F1"],   df_results.loc[1, "Non0_F1"],   df_results.loc[3, "Has0_F1"],   df_results.loc[3, "Non0_F1"],   df_results.loc[5, "Has0_F1"],   df_results.loc[5, "Non0_F1"]]

plot_metric_comparison(acc_base, acc_cot, "Accuracy (Acc-2, %)", (72.0, 91.5), "acc2_comparison_paper")
plot_metric_comparison(f1_base,  f1_cot,  "F1-Score (%)",        (70.0, 90.0), "f1_comparison_paper")